[스택 오버플로우 참조](https://stackoverflow.com/questions/55460434/how-to-export-save-an-animated-bubble-chart-made-with-plotly)

In [8]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import io
import PIL

In [9]:
nyc_landmarks = {
    "Name": ["Wall Street", "Midtown Manhattan", "Times Square", 
             "Central Park", "Statue of Liberty", "Forest Park", "Citi Field"],
    "Latitude": [40.7074, 40.7549, 40.7580, 40.785091, 40.6892, 40.7028, 40.7571],
    "Longitude": [-74.0113, -73.9840, -73.9855, -73.968285, -74.0445, -73.8495, -73.8458]
}

df_nyc_landmarks = pd.DataFrame(nyc_landmarks)

df = pd.read_csv("https://raw.githubusercontent.com/guebin/DV2023/main/posts/NYCTaxi.csv")

In [10]:
df_feature = df.assign(
    log_trip_duration = np.log(df.trip_duration),  ## 로그변환
    pickup_datetime = pd.to_datetime(df.pickup_datetime),  ## datetime 형식으로 변환
    dropoff_datetime = pd.to_datetime(df.dropoff_datetime),
    dist = np.sqrt((df.pickup_latitude - df.dropoff_latitude)**2 + (df.pickup_longitude - df.dropoff_longitude)**2),
    #---#
    vendor_id = df.vendor_id.map({1:'A', 2:'B'})  ## 범주형으로 명시, 딕셔너리와 map()을 이용
).assign(
    speed = lambda _df : _df.dist / _df.trip_duration,
    pickup_hour = lambda _df : _df.pickup_datetime.dt.hour,  ## 탑승 시간을 할당
    dropoff_hour = lambda _df : _df.dropoff_datetime.dt.hour,
    dayofweek = lambda _df : _df.pickup_datetime.dt.dayofweek  ## 탑승한 시점으로 요일을 잡음
)

In [11]:
fig = px.scatter_mapbox(
    data_frame = df_feature.sort_values('pickup_hour'), ## 인덱스 순서가 제대로 되도록...
    lat = 'pickup_latitude',
    lon = 'pickup_longitude',
    color = 'vendor_id',
    size = 'passenger_count', size_max = 5,
    animation_frame = 'pickup_hour',
    center = {'lat' : 40.7322, 'lon' : -73.9052},
    #---#
    mapbox_style = 'carto-positron',
    zoom = 10,
    width = 750,
    height = 600
)

fig.show(config = {'scrollZoom' : False})

/tmp/ipykernel_60239/2225368957.py:1: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [40]:
fig.update_layout(
    title_x = 0.5,
    title_y = 0.95,
    title = "시간별 뉴욕시의 택시 승차 장소 시각화(Kaggle Dataset)",
    legend = {"title":"업체명","traceorder":"reversed"}
)

In [ ]:
# import pickle

# with open(file = "fig.pkl", mode = "wb") as f :
#     pickle.dump(fig, f)
    
# with open(file = "fig.pkl", mode = "rb") as f :
#     fig = pickle.load(f)

In [41]:
frames = []

for s, fr in enumerate(fig.frames) :
    fig.update(data = fr.data)
    fig.layout.sliders[0].update(active = s)
    frames.append(PIL.Image.open(io.BytesIO(fig.to_image(format = "png"))))
    print(f"layer {s} is appended...")
    
frames[0].save(
    "test.gif",
    save_all = True,
    append_images = frames[1:],
    optimize = True,
    duration = 500,
    loop = 0
)

layer 0 is appended...
layer 1 is appended...
layer 2 is appended...
layer 3 is appended...
layer 4 is appended...
layer 5 is appended...
layer 6 is appended...
layer 7 is appended...
layer 8 is appended...
layer 9 is appended...
layer 10 is appended...
layer 11 is appended...
layer 12 is appended...
layer 13 is appended...
layer 14 is appended...
layer 15 is appended...
layer 16 is appended...
layer 17 is appended...
layer 18 is appended...
layer 19 is appended...
layer 20 is appended...
layer 21 is appended...
layer 22 is appended...
layer 23 is appended...
